In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

# Load Data (JSONLs and Extracted Features Parquet)

In [ ]:
import polars as pl
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent 
DATA_DIR = PROJECT_ROOT / "data"
EXPLORATION_DIR = PROJECT_ROOT / "exploration"
PARQUET_FILE = EXPLORATION_DIR / "engineered_audio_features.parquet"

df = pl.read_parquet(PARQUET_FILE)

df.head()

# File analysis

In [ ]:
summary = df.group_by("source_file").agg([
    pl.len().alias("total_utterances"),
    pl.col("error").is_not_null().sum().alias("file_not_found"),
    pl.col("error").is_null().sum().alias("successful_audio")
])

display(summary)

print(f"Ammount of succesful tracks: {summary['successful_audio'].sum()}")

Remove File not found entries

In [ ]:
df = df.filter(pl.col("error").is_null())

In [ ]:
# Total rows that are part of a duplicate group
total_duplicate_rows = df.filter(pl.col('utterance_id').is_duplicated()).height

# Extra redundant rows to drop (total rows minus unique utterance_ids)
extra_redundant_rows = df.height - df.select('utterance_id').n_unique()

print(f"Total rows involved in duplicates: {total_duplicate_rows}")
print(f"Number of extra rows to drop (if deduplicating): {extra_redundant_rows}")

# 

# Metadata (age, duration)

In [ ]:
sns.set_theme(style="whitegrid")

# Select the columns, drop missing values, and push to Pandas for Seaborn
plot_df = df.select(['audio_duration_sec', 'age_bucket']).drop_nulls().to_pandas()
count = (df["audio_duration_sec"] > 0).sum()
print(count)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Histogram of Audio Duration
sns.histplot(data=plot_df, x='audio_duration_sec', bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Audio Duration (Sec)', fontweight='bold')
axes[0].set_xlabel('Seconds')
axes[0].set_ylabel('Count')

# Plot 2: Histogram / Count of Age Buckets
order = ['3-4', '5-7', '8-11', '12+', 'unknown']
sns.countplot(
    data=plot_df,
    x='age_bucket',
    order=order,
    ax=axes[1],
    color='lightcoral'
)
axes[1].set_title('Count of Files per Age Bucket', fontweight='bold')
axes[1].set_xlabel('Age Bucket')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("Stats on {Audio Duration}:")
display(df['audio_duration_sec'].describe())

In [ ]:
# Calculate total hours of speech per age group
hours_per_age = (
    df.group_by('age_bucket')
    .agg((pl.col('audio_duration_sec').sum() / 3600).alias('total_hours'))
    .sort('total_hours', descending=True)
)

print("Total Hours per Age Group:")
display(hours_per_age)

# Plot the total hours
plt.figure(figsize=(10, 5))
sns.barplot(
    data=hours_per_age.to_pandas(), 
    x='age_bucket', 
    y='total_hours', 
    palette='viridis'
)
plt.title('Total Hours of Speech per Age Group', fontweight='bold')
plt.xlabel('Age Bucket')
plt.ylabel('Total Hours')
plt.show()

# Speaker and Session Diversity

In [ ]:
# Calculate unique counts
unique_children = df.select(pl.col('child_id').n_unique()).item()
unique_sessions = df.select(pl.col('session_id').n_unique()).item()

print(f"Total Unique children (child_id): {unique_children}")
print(f"Total Unique sessions (session_id): {unique_sessions}")

# Group by child_id and count unique sessions
child_session_check = (
    df.group_by('child_id')
    .agg(pl.col('session_id').n_unique().alias('unique_session_count'))
)

# Filter for children who have MORE than 1 session
multi_session_children = child_session_check.filter(pl.col('unique_session_count') > 1)

# Check the results
if multi_session_children.height == 0:
    print("\nEvery child_id corresponds to exactly ONE session_id.")
else:
    print(f"\nAlert: {multi_session_children.height} child_ids map to MULTIPLE session_ids.")
    print("Here are the worst offenders:")
    display(multi_session_children.sort('unique_session_count', descending=True).head(5))

In [ ]:
# Group by child to count unique sessions
sessions_per_child = (
    df.group_by('child_id')
    .agg(pl.col('utterance_id').n_unique().alias('utterance_count'))
    .sort('utterance_count', descending=True)
)

# Plot the distribution
plt.figure(figsize=(10, 5))
sns.histplot(
    sessions_per_child.to_pandas(), 
    x='utterance_count', 
    bins=range(1, sessions_per_child['utterance_count'].max() + 2), 
    color='purple',
    discrete=True
)
plt.title('Distribution of Utterances per Child', fontweight='bold')
plt.xlabel('Number of utterances')
plt.ylabel('Number of Children')
plt.show()

print("Describe utterance_count")
display(sessions_per_child['utterance_count'].describe())

In [ ]:
# Group by child to count unique sessions
sessions_per_child = (
    df.group_by('child_id')
    .agg(pl.col('session_id').n_unique().alias('session_count'))
    .sort('session_count', descending=True)
)

# Plot the distribution
plt.figure(figsize=(10, 5))
sns.histplot(
    sessions_per_child.to_pandas(), 
    x='session_count', 
    bins=range(1, sessions_per_child['session_count'].max() + 2), 
    color='purple',
    discrete=True
)
plt.title('Distribution of Sessions per Child', fontweight='bold')
plt.xlabel('Number of Unique Sessions')
plt.ylabel('Number of Children')
plt.show()

print("Describe session count")
display(sessions_per_child['session_count'].describe())

In [ ]:
# Group by session_id and count unique children
children_per_session = (
    df.group_by('session_id')
    .agg(pl.col('child_id').n_unique().alias('child_count'))
    .filter(pl.col('child_count') > 1)
    .sort('child_count', descending=True)
)

if children_per_session.height > 0:
    print(f"{children_per_session.height} sessions contain MULTIPLE children.")
    display(children_per_session.head())
else:
    print("Each session belongs to exactly ONE child. It's a strict 1-to-Many mapping (Child -> Sessions).")

In [ ]:
# Check for children crossing age buckets
age_changes = (
    df.group_by('child_id')
    .agg(
        pl.col('age_bucket').n_unique().alias('unique_age_buckets'),
        pl.col('age_bucket').drop_nulls().unique().alias('ages_listed')
    )
    .filter(pl.col('unique_age_buckets') > 1)
)

if age_changes.height > 0:
    print(f"{age_changes.height} children have sessions across different age buckets.")
    display(age_changes.head(5))
else:
    print("All multiple sessions for a child occurred within the same age bucket.")

# age_changes.describe()

# Labels

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Count all tokens in the phonetic transcriptions
phoneme_counts = Counter()
total_tokens = 0

for path in found_paths:
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            tokens = obj.get('phonetic_text', '').split()
            phoneme_counts.update(tokens)
            total_tokens += len(tokens)

print(f"Total tokens processed: {total_tokens}")
print(f"Unique phonemes found: {len(phoneme_counts)}")

# Convert to DataFrame for easier handling and plotting
df_phonemes = pd.DataFrame(
    phoneme_counts.most_common(50), 
    columns=['phoneme', 'count']
)

# Display top 10
print("\nTop 10 Most Common Phonemes:")
display(df_phonemes.head(10))

# Plot the top 30 phonemes
plt.figure(figsize=(14, 6))
sns.barplot(data=df_phonemes.head(30), x='phoneme', y='count', palette='muted')
plt.title('Top 30 Most Frequent Phonemes in Corpus', fontweight='bold')
plt.xlabel('Phoneme')
plt.ylabel('Frequency Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Audio Metadata

In [ ]:
import matplotlib.pyplot as plt

# 1. Stereo vs Mono Distribution
stereo_counts = (
    df.group_by('is_stereo')
    .len(name='count')
    .sort('count', descending=True)
)

# 2. Sample Rate Distribution
sr_counts = (
    df.group_by('native_sample_rate')
    .len(name='count')
    .sort('count', descending=True)
)

# 3. Bit Depth Distribution
bit_depth_counts = (
    df.group_by('bit_depth')
    .len(name='count')
    .sort('count', descending=True)
)

# Extract data to standard python lists for plotting
sr_data = sr_counts.to_dicts()
bd_data = bit_depth_counts.to_dicts()
stereo_data = stereo_counts.to_dicts()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Sample Rates
sr_labels = [str(row['native_sample_rate']) for row in sr_data]
sr_vals = [row['count'] for row in sr_data]
axes[0].bar(sr_labels, sr_vals, color='#4CAF50', edgecolor='black')
axes[0].set_title('Native Sample Rates')
axes[0].set_xlabel('Sample Rate (Hz)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Bit Depths
bd_labels = [str(row['bit_depth']) for row in bd_data]
bd_vals = [row['count'] for row in bd_data]
axes[1].bar(bd_labels, bd_vals, color='#2196F3', edgecolor='black')
axes[1].set_title('Bit Depths')
axes[1].set_xlabel('Bit Depth')

# Plot 3: Stereo vs Mono
st_labels = ['Stereo' if row['is_stereo'] else 'Mono' for row in stereo_data]
st_vals = [row['count'] for row in stereo_data]
axes[2].pie(st_vals, labels=st_labels, autopct='%1.1f%%', colors=['#FF9800', '#9E9E9E'], startangle=90)
axes[2].set_title('Stereo vs Mono')

plt.tight_layout()
plt.show()

# DSP Analysis

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

# Set a nice theme for the plots
sns.set_theme(style="whitegrid")

# Select the DSP features and drop nulls for plotting
dsp_cols = ['clipping_amount', 'zcr_mean', 'centroid_mean', 'f0_mean']
dsp_df = df.select(dsp_cols).drop_nulls().to_pandas()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(dsp_cols):
    sns.histplot(dsp_df[col], bins=50, kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Select relevant columns, drop nulls, and convert to pandas
age_dsp_df = df.select(['age_bucket'] + dsp_cols).drop_nulls().to_pandas()

# Sort the age buckets if they have a natural order (optional)
# age_dsp_df = age_dsp_df.sort_values('age_bucket')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(dsp_cols):
    sns.boxplot(data=age_dsp_df, x='age_bucket', y=col, ax=axes[i], palette='Set2', hue='age_bucket')
    axes[i].set_title(f'{col} by Age Bucket')
    axes[i].set_xlabel('Age Bucket')
    axes[i].set_ylabel(col)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
import math

# 1. Get unique age buckets (dropping nulls to keep the plots clean, but you can remove drop_nulls() if you want a pie chart for missing ages too)
unique_ages = df.select('age_bucket').drop_nulls().unique().to_series().to_list()

# 2. Set up the subplot grid
n_buckets = len(unique_ages)
cols = 3  
rows = math.ceil(n_buckets / cols)

# Create the figure and axes
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = axes.flatten() #

# 3. Loop through each age bucket and create its pie chart
for i, age in enumerate(unique_ages):
    ax = axes[i]
    
    clipping_summary = (
        df.filter(pl.col('age_bucket') == age)
        .with_columns((pl.col('clipping_amount') > 1.0).alias('high_clipping'))
        .group_by('high_clipping')
        .len(name='count')
    )
    
    counts_dict = dict(zip(clipping_summary['high_clipping'], clipping_summary['count']))
    
    labels = []
    sizes = []
    colors = []
    explode = []
    
    if False in counts_dict:
        labels.append("Clipping <= 1")
        sizes.append(counts_dict[False])
        colors.append("#4CAF50")
        explode.append(0)
        
    if True in counts_dict:
        labels.append("Clipping > 1")
        sizes.append(counts_dict[True])
        colors.append("#F44336")
        explode.append(0.1) # Pop out the high clipping slice
        
    if sizes: 
        ax.pie(
            sizes, 
            labels=labels, 
            colors=colors, 
            explode=explode,
            autopct='%1.1f%%', 
            startangle=140, 
            textprops={'fontsize': 10}
        )
        ax.set_title(f'Age Bucket: {age}', fontsize=12, fontweight='bold')
    else:
        # Hide the axis if for some reason there's no data
        ax.set_visible(False)

# 4. Hide any extra empty subplots (if n_buckets is not a perfect multiple of cols)
for j in range(len(unique_ages), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Proportion of Audio Files with Clipping > 1 by Age Bucket', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Combined Data (Original + Your New Data)
beam_widths = ['1', '5', '10', '15', '20', '30','50']
time = [282, 296, 322, 335, 355, 404,477]
score = [0.3539, 0.3491, 0.3479, 0.3475, 0.3474, 0.3471,0.3471 ]

x = np.arange(len(beam_widths))  # The label locations
width = 0.5  # The width of the bars

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Time (Bar chart on left y-axis)
rects = ax1.bar(x, time, width, label='Time', color='skyblue', edgecolor='black', alpha=0.8)
ax1.set_xlabel('Beam Width', fontweight='bold')
ax1.set_ylabel('Time', color='steelblue', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_ylim(0, 450) # Increased max limit to accommodate '404'
ax1.grid(axis='y', linestyle='--', alpha=0.4)

# Add labels to bars
for rect in rects:
    height = rect.get_height()
    ax1.annotate(f'{height}',
                 xy=(rect.get_x() + rect.get_width() / 2, height),
                 xytext=(0, 3),  # 3 points vertical offset
                 textcoords="offset points",
                 ha='center', va='bottom', color='steelblue', fontweight='bold')

# Plot Score (Line chart on right y-axis)
ax2 = ax1.twinx()  # Instantiate a second axes that shares the same x-axis
line = ax2.plot(x, score, label='Score', color='#FF6700', marker='o', linewidth=2.5)
ax2.set_ylabel('Score', color='#FF6700', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#FF6700')
ax2.set_ylim(0.345, 0.356) 

# Add labels to line points
for i, txt in enumerate(score):
    ax2.annotate(f'{txt:.4f}',
                 xy=(x[i], score[i]),
                 xytext=(0, 8),  # 8 points vertical offset
                 textcoords="offset points",
                 ha='center', va='bottom', color="#FF6700", fontweight='bold')

# Add x-axis labels
ax1.set_xticks(x)
ax1.set_xticklabels(beam_widths)

# Title and Legend
plt.title('Beam Width vs. Time and Score Val \n utterances:   30613', fontweight='bold', pad=15)

# Combine legends from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

plt.tight_layout()
plt.show()

## Length Mark Previous-Token Analysis

Find tokens immediately *before* length-marked tokens (ː or ˑ), then summarize frequency and probability.

In [ ]:
from collections import Counter
import json
from pathlib import Path

# Resolve repo root so paths work when running from notebooks in subfolders

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'data').exists():
            return p
    return start

repo_root = find_repo_root(Path.cwd())

data_paths = [
    repo_root / 'data' / 'train_phon_transcripts.jsonl',
    repo_root / 'data' / 'train_phon_transcripts_talkbank.jsonl',
    repo_root / 'data' / 'train_phon_transcripts_ultrasuite.jsonl',
]

found_paths = [p for p in data_paths if p.exists()]
if not found_paths:
    raise FileNotFoundError(
        'No transcript files found. Expected one of: ' + ', '.join(str(p) for p in data_paths)
    )
if len(found_paths) < len(data_paths):
    missing = [p for p in data_paths if not p.exists()]
    print('Warning: missing files:', ', '.join(str(p) for p in missing))


# def analyze_prev(path: Path):
#     prev_counts = Counter()
#     mark_type_counts = Counter()
#     total_sequences = 0
#     total_mark_tokens = 0
#     with path.open('r', encoding='utf-8') as f:
#         for line in f:
#             obj = json.loads(line)
#             tokens = obj.get('phonetic_text', '').split()
#             for i, tok in enumerate(tokens):
#                 if 'ː' in tok or 'ˑ' in tok:
#                     total_mark_tokens += 1
#                     if 'ː' in tok: mark_type_counts['ː'] += 1
#                     if 'ˑ' in tok: mark_type_counts['ˑ'] += 1
#                     if i - 1 >= 0:
#                         prev_counts[tokens[i - 1]] += 1
#                         total_sequences += 1
#     return prev_counts, total_sequences, total_mark_tokens, mark_type_counts


# results = {}
# for path in found_paths:
#     results[path.name] = analyze_prev(path)

# combined_prev = Counter()
# combined_total_sequences = 0
# combined_total_mark_tokens = 0
# combined_mark_types = Counter()

# for prev_counts, total_sequences, total_mark_tokens, mark_types in results.values():
#     combined_prev.update(prev_counts)
#     combined_total_sequences += total_sequences
#     combined_total_mark_tokens += total_mark_tokens
#     combined_mark_types.update(mark_types)

# print('Combined sequences:', combined_total_sequences)
# print('Combined length-mark tokens:', combined_total_mark_tokens)
# print('Combined mark types:', dict(combined_mark_types))

# print('Top 20 previous tokens:')
# for tok, count in combined_prev.most_common(20):
#     prob = count / combined_total_sequences if combined_total_sequences else 0.0
#     print(tok, count, f'{prob:.4f}')

# # Write outputs
# out_dir = repo_root / 'outputs'
# out_dir.mkdir(parents=True, exist_ok=True)
# out_path = out_dir / 'length_mark_prev_token.csv'
# with out_path.open('w', encoding='utf-8') as f:
#     f.write('token,count,probability
# ')
#     for tok, count in combined_prev.most_common():
#         prob = count / combined_total_sequences if combined_total_sequences else 0.0
#         f.write(f'{tok},{count},{prob:.8f}
# ')

# per_path = out_dir / 'length_mark_prev_token_by_dataset.csv'
# with per_path.open('w', encoding='utf-8') as f:
#     f.write('dataset,token,count,probability
# ')
#     for name, (prev_counts, total_sequences, _, _) in results.items():
#         for tok, count in prev_counts.most_common():
#             prob = count / total_sequences if total_sequences else 0.0
#             f.write(f'{name},{tok},{count},{prob:.8f}
# ')

# seq_path = out_dir / 'length_mark_prev_sequences.csv'
# with seq_path.open('w', encoding='utf-8') as f:
#     f.write('dataset,utterance_id,prev_token,mark_token,token_index
# ')
#     for path in found_paths:
#         with path.open('r', encoding='utf-8') as f_in:
#             for line in f_in:
#                 obj = json.loads(line)
#                 tokens = obj.get('phonetic_text', '').split()
#                 for i, tok in enumerate(tokens):
#                     if 'ː' in tok or 'ˑ' in tok:
#                         if i - 1 >= 0:
#                             f.write(f'{path.name},{obj.get("utterance_id","")},{tokens[i - 1]},{tok},{i}
# ')


import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Count all individual characters in the phonetic transcriptions
phoneme_counts = Counter()
total_characters = 0

for path in found_paths:
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            # Remove spaces and get individual characters
            characters = list(obj.get('phonetic_text', '').replace(' ', ''))
            phoneme_counts.update(characters)
            total_characters += len(characters)

print(f"Total characters processed: {total_characters}")
print(f"Unique phoneme characters found: {len(phoneme_counts)}")

# Convert to DataFrame for easier handling and plotting
df_phonemes = pd.DataFrame(
    phoneme_counts.most_common(55), 
    columns=['phoneme', 'count']
)

# Display top 10
print("\nPhoneme Characters Count:")
display(df_phonemes)

# Plot the top 30 phonemes
plt.figure(figsize=(14, 6))
sns.barplot(data=df_phonemes.head(30), x='phoneme', y='count', palette='muted')
plt.title('Top 30 Most Frequent Phoneme Characters in Corpus', fontweight='bold')
plt.xlabel('Phoneme Character')
plt.ylabel('Frequency Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()